# E05_equipeRRB_qualidade_dados
## Entrega 5 — Obtenção, limpeza e tratamento dos dados

**Projeto:** indicadores econômicos da Paraíba  
**Equipe:** Maria Beatriz Victoria Larangeira; Maria Rayanne Martins Batista; Roberta Gabriela Moreira Bastos.

### Objetivo
Construir uma base analítica reproduzível a partir dos arquivos do ZIP, preservar os originais, diagnosticar a qualidade, tratar PIB e indicadores conjunturais, extrair a arrecadação mensal de ICMS quando o documento contém informação legível, validar e exportar os resultados.

**Importante:** a base municipal do PIB disponibilizada pelo IBGE contém 223 municípios e uma linha de total estadual. Alguns PDFs de ICMS estão digitalizados como imagem e/ou não apresentam o quadro de total em todas as páginas. O notebook não inventa valores: meses não extraídos ficam identificados como pendentes.

## 1. Bibliotecas e pastas

In [1]:
import os, re, io, csv, zipfile, shutil, unicodedata, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 180)

ROOT = Path("E05")
RAW = ROOT / "data" / "raw"
EXT = RAW / "extraidos"
PROC = ROOT / "data" / "processed"
DOCS = ROOT / "docs"
for p in [RAW, EXT, PROC, DOCS]:
    p.mkdir(parents=True, exist_ok=True)

ANOS = list(range(2010, 2024))
EXECUCAO = datetime.now().isoformat(timespec="seconds")
print("Estrutura pronta:", ROOT.resolve())

Estrutura pronta: /content/E05


## 2. Enviar e extrair o ZIP
Selecione o arquivo `todos os dados...zip` enviado pela equipe. O ZIP original fica preservado.

In [2]:
from google.colab import files
upload = files.upload()
zip_files = [n for n in upload if n.lower().endswith(".zip")]
if not zip_files:
    raise ValueError("Nenhum ZIP foi enviado.")
zip_name = zip_files[0]
zip_path = RAW / Path(zip_name).name
zip_path.write_bytes(upload[zip_name])

if EXT.exists():
    shutil.rmtree(EXT)
EXT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(EXT)

all_files = [p for p in EXT.rglob("*") if p.is_file()]
print("ZIP preservado:", zip_path)
print("Arquivos extraídos:", len(all_files))

Saving todos  os dados-20261105T232855Z-1-001.zip to todos  os dados-20261105T232855Z-1-001.zip
ZIP preservado: E05/data/raw/todos  os dados-20261105T232855Z-1-001.zip
Arquivos extraídos: 22


## 3. Inventário e diagnóstico dos arquivos

In [3]:
inventario = pd.DataFrame([{
    "arquivo": p.name, "caminho": str(p.relative_to(EXT)),
    "extensao": p.suffix.lower(), "tamanho_bytes": p.stat().st_size
} for p in all_files]).sort_values("caminho").reset_index(drop=True)
display(inventario)
inventario.to_csv(DOCS/"inventario_arquivos.csv", index=False, encoding="utf-8-sig")

csv_paths = sorted(EXT.rglob("*.csv"))
pdf_paths = sorted(EXT.rglob("*.pdf"))
excel_paths = sorted(list(EXT.rglob("*.xls")) + list(EXT.rglob("*.xlsx")))
print(f"CSV: {len(csv_paths)} | PDF: {len(pdf_paths)} | Excel: {len(excel_paths)}")

,arquivo,caminho,extensao,tamanho_bytes
0,4-tabelas.xls,todos os dados/4-tabelas.xls,.xls,510464
1,ICMS2015.pdf,todos os dados/ICMS2015.pdf,.pdf,87065
2,ICMS2016.pdf,todos os dados/ICMS2016.pdf,.pdf,83909
3,ICMS2017.pdf,todos os dados/ICMS2017.pdf,.pdf,83594
4,ICMS2018.pdf,todos os dados/ICMS2018.pdf,.pdf,3374034
5,ICMS2019.pdf,todos os dados/ICMS2019.pdf,.pdf,3075973
6,ICMS2020.pdf,todos os dados/ICMS2020.pdf,.pdf,3596146
7,ICMS2021.pdf,todos os dados/ICMS2021.pdf,.pdf,89184
8,ICMS2022.pdf,todos os dados/ICMS2022.pdf,.pdf,110968
9,ICMS2023.pdf,todos os dados/ICMS2023.pdf,.pdf,117268


CSV: 8 | PDF: 12 | Excel: 2


## 4. Funções de leitura e conversão

Os CSVs do ZIP não possuem um único padrão. Cada base será lida com separador e linha de cabeçalho identificados na inspeção do arquivo. A base municipal tem campos vazios de unidade entre os anos; por isso a leitura usa `index_col=False` para não deslocar as três colunas de identificação.

In [4]:
def ler_texto(path):
    raw = path.read_bytes()
    for enc in ["utf-8-sig", "latin-1"]:
        try: return raw.decode(enc)
        except UnicodeDecodeError: pass
    return raw.decode("utf-8", errors="replace")

def ptbr_num(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().replace("%","")
    if s in ("", "-", "...", "..", "X", "nan", "None"): return np.nan
    # vírgula decimal; pontos de milhar
    if "," in s: s = s.replace(".","").replace(",",".")
    return pd.to_numeric(s, errors="coerce")

def normalizar_nome(s):
    if pd.isna(s): return np.nan
    s = re.sub(r"\s*\(PB\)\s*$", "", str(s).strip(), flags=re.I)
    s = unicodedata.normalize("NFKD", s).encode("ascii","ignore").decode("ascii")
    return re.sub(r"\s+"," ",s).upper()

def achar(termo):
    encontrados = [p for p in csv_paths if termo.lower() in p.name.lower()]
    if len(encontrados) != 1:
        raise ValueError(f"Esperava 1 arquivo para '{termo}', encontrei {[p.name for p in encontrados]}")
    return encontrados[0]

## 5. Tratamento do PIB Brasil, estados e Paraíba

In [5]:
pib_brasil_path = achar("PIBbrasil")
pib_estados_path = achar("PIBestados")
pib_pb_path = achar("PIBparaíba")

def ler_pib_agregado(path, nome_coluna):
    # Linhas 0–2 são título, variável e identificação; linha 3 contém os nomes dos anos.
    df = pd.read_csv(path, sep=",", header=3, dtype=str, encoding="utf-8-sig",
                     engine="python", index_col=False)
    df = df.rename(columns={df.columns[0]: nome_coluna})
    anos = [str(a) for a in ANOS if str(a) in df.columns]
    for a in anos: df[a] = df[a].map(ptbr_num)
    df[nome_coluna] = df[nome_coluna].astype(str).str.strip()
    df = df[df[nome_coluna].ne("") & df[nome_coluna].str.lower().ne("nan")]
    return df.reset_index(drop=True)

pib_brasil = ler_pib_agregado(pib_brasil_path, "unidade")
pib_estados = ler_pib_agregado(pib_estados_path, "unidade_federacao")
pib_paraiba = ler_pib_agregado(pib_pb_path, "unidade_federacao")

print("PIB Brasil:", pib_brasil.shape)
print("PIB estados:", pib_estados.shape)
print("PIB Paraíba:", pib_paraiba.shape)
display(pib_brasil)
display(pib_estados)
display(pib_paraiba)

PIB Brasil: (14, 15)
PIB estados: (40, 15)
PIB Paraíba: (14, 15)


,unidade,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,Brasil,3.885847e+09,4.376382e+09,4.814760e+09,5.331619e+09,5.778953e+09,5.995787e+09,6.269328e+09,6.585479e+09,7.004141e+09,7.389131e+09,7.609597e+09,9.012142e+09,1.007968e+10,1.094335e+10
1,"Fonte: IBGE, em parceria com os Órgãos Estadua...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Notas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1 - Os dados de 2022 e 2023 estarão sujeitos a...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2 - Para os anos de 2022 e 2023 foram divulgad...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,3 - Os dados da série retropolada (de 2002 a 2...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Legenda,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Símbolo,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,unidade_federacao,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,Rondônia,2.390789e+07,2.757471e+07,3.011272e+07,3.112141e+07,3.403098e+07,3.656333e+07,3.946036e+07,4.351615e+07,4.491398e+07,4.709134e+07,5.159874e+07,5.817010e+07,6.679545e+07,7.645618e+07
1,Acre,8.342356e+06,8.949434e+06,1.013792e+07,1.147393e+07,1.345870e+07,1.362280e+07,1.375424e+07,1.427294e+07,1.533112e+07,1.563002e+07,1.647637e+07,2.137444e+07,2.367614e+07,2.629132e+07
2,Amazonas,6.087712e+07,7.073440e+07,7.224270e+07,8.305123e+07,8.666864e+07,8.656818e+07,8.903978e+07,9.324019e+07,1.001092e+08,1.081811e+08,1.160191e+08,1.315310e+08,1.451405e+08,1.617950e+08
3,Roraima,6.639150e+06,7.303719e+06,7.711467e+06,9.010725e+06,9.744122e+06,1.024290e+07,1.101324e+07,1.210471e+07,1.336999e+07,1.429223e+07,1.602428e+07,1.820258e+07,2.109534e+07,2.512480e+07
4,Pará,8.268452e+07,9.871074e+07,1.070809e+08,1.212248e+08,1.245849e+08,1.308995e+08,1.381075e+08,1.552324e+08,1.613496e+08,1.783770e+08,2.159356e+08,2.629050e+08,2.361419e+08,2.545465e+08
5,Amapá,8.237795e+06,9.409228e+06,1.113087e+07,1.276349e+07,1.340028e+07,1.386129e+07,1.434214e+07,1.548191e+07,1.679521e+07,1.749666e+07,1.846912e+07,2.009985e+07,2.361429e+07,2.802012e+07
6,Tocantins,1.640482e+07,1.834569e+07,2.068443e+07,2.379666e+07,2.618932e+07,2.893029e+07,3.158482e+07,3.410813e+07,3.566618e+07,3.935594e+07,4.364980e+07,5.178076e+07,5.820880e+07,6.431770e+07
7,Maranhão,4.630963e+07,5.214354e+07,6.049011e+07,6.769484e+07,7.684203e+07,7.847599e+07,8.531004e+07,8.954276e+07,9.817950e+07,9.733994e+07,1.069160e+08,1.249807e+08,1.397891e+08,1.492272e+08
8,Piauí,2.226915e+07,2.594136e+07,2.863768e+07,3.128359e+07,3.772350e+07,3.914969e+07,4.141694e+07,4.536554e+07,5.037842e+07,5.278078e+07,5.639126e+07,6.402830e+07,7.283450e+07,8.091686e+07
9,Ceará,7.933630e+07,8.969583e+07,9.697375e+07,1.090366e+08,1.260545e+08,1.306298e+08,1.384225e+08,1.479215e+08,1.559038e+08,1.635753e+08,1.669145e+08,1.948848e+08,2.136007e+08,2.322393e+08


,unidade_federacao,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,Paraíba,3.352249e+07,3.710914e+07,42488349.2,4.637730e+07,5.293648e+07,5.614189e+07,5.910478e+07,6.239678e+07,6.437360e+07,67986073.55,7.029203e+07,7.747033e+07,8.609417e+07,9.696317e+07
1,"Fonte: IBGE, em parceria com os Órgãos Estadua...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Notas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1 - Os dados de 2022 e 2023 estarão sujeitos a...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2 - Para os anos de 2022 e 2023 foram divulgad...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,3 - Os dados da série retropolada (de 2002 a 2...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Legenda,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Símbolo,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6. Tratamento correto do PIB municipal da Paraíba

A fonte possui 237 linhas: 223 municípios, 1 total estadual e linhas de fonte/notas/legenda. Os anos ocupam colunas alternadas com colunas de unidade. Mantemos apenas os registros `MU` e `UF`, selecionamos os valores anuais e registramos que a unidade da fonte é Mil Reais.

In [6]:
pib_mun_path = achar("PIBmunicípiosPB")
raw_mun = pd.read_csv(pib_mun_path, sep=";", header=3, dtype=str,
                      encoding="utf-8-sig", engine="python", index_col=False)

print("Dimensão bruta:", raw_mun.shape)
print("Tipos de linha:", raw_mun.iloc[:,0].value_counts(dropna=False).to_dict())

pib_mun = pd.DataFrame({
    "nivel": raw_mun.iloc[:,0].astype(str).str.strip(),
    "codigo_ibge": raw_mun.iloc[:,1].astype(str).str.strip(),
    "municipio": raw_mun.iloc[:,2].astype(str).str.strip()
})
for ano in ANOS:
    col = str(ano)
    if col in raw_mun.columns:
        pib_mun[col] = raw_mun[col].map(ptbr_num)
    else:
        pib_mun[col] = np.nan

pib_mun = pib_mun[pib_mun["nivel"].isin(["UF","MU"])].copy()
pib_mun["codigo_ibge"] = pib_mun["codigo_ibge"].str.extract(r"(\d{7}|25)", expand=False)
pib_mun["municipio_padronizado"] = pib_mun["municipio"].map(normalizar_nome)
pib_mun["unidade"] = "Mil Reais"
pib_mun = pib_mun.reset_index(drop=True)

municipios = pib_mun[pib_mun["nivel"].eq("MU")].copy()
total_pb_mun = pib_mun[pib_mun["nivel"].eq("UF")].copy()

print("Registros municipais:", len(municipios))
print("Total estadual na tabela municipal:", len(total_pb_mun))
print("Códigos municipais únicos:", municipios["codigo_ibge"].nunique())
print("Municípios com código ausente:", municipios["codigo_ibge"].isna().sum())
display(municipios)

Dimensão bruta: (237, 30)
Tipos de linha: {'MU': 223, 'UF': 1, 'Fonte: IBGE, em parceria com os Órgãos Estaduais de Estatística, Secretarias Estaduais de Governo e Superintendência da Zona Franca de Manaus - SUFRAMA': 1, 'Notas': 1, '1 - Os dados de 2022 e 2023 estarão sujeitos a revisão na próxima publicação.': 1, '2 - Para os anos de 2022 e 2023 foram divulgados apenas o Produto Interno Bruto e o Produto Interno Bruto per capita, conforme a nota técnica 02/2024 do Sistema de Contas Nacionais, que pode ser acessada em https://biblioteca.ibge.gov.br/visualizacao/livros/liv102094.pdf.': 1, '3 - Os dados da série retropolada (de 2002 a 2009) também têm como referência o ano de 2010, seguindo a nova referência das Contas Nacionais.': 1, 'Legenda': 1, 'Símbolo': 1, '-': 1, '0': 1, 'X': 1, '..': 1, '...': 1, 'A a Z\n(exceto X)': 1}
Registros municipais: 223
Total estadual na tabela municipal: 1
Códigos municipais únicos: 223
Municípios com código ausente: 0


/tmp/ipykernel_2434/2278224658.py:2: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  raw_mun = pd.read_csv(pib_mun_path, sep=";", header=3, dtype=str,


,nivel,codigo_ibge,municipio,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,municipio_padronizado,unidade
1,MU,2500106,Água Branca (PB),40110.294,44488.957,49023.325,54926.174,64602.600,70799.309,75098.991,77222.249,79901.459,82544.251,86555.838,92268.329,95192.995,107368.213,AGUA BRANCA,Mil Reais
2,MU,2500205,Aguiar (PB),25767.837,28877.221,31362.405,35153.449,37810.694,39794.974,44335.610,47394.371,46220.879,50141.432,55808.344,60255.576,61115.323,68585.332,AGUIAR,Mil Reais
3,MU,2500304,Alagoa Grande (PB),139059.942,164280.838,172724.437,195412.499,211284.286,225940.710,239624.742,266622.078,256889.736,276069.189,281480.230,300766.384,320351.096,367231.922,ALAGOA GRANDE,Mil Reais
4,MU,2500403,Alagoa Nova (PB),139937.639,177696.303,167735.949,237754.430,223174.808,257150.215,250823.904,225944.787,287713.251,325789.766,331770.266,335568.446,377884.689,376825.036,ALAGOA NOVA,Mil Reais
5,MU,2500502,Alagoinha (PB),56027.306,63841.924,72202.497,71853.975,83955.115,91454.214,99535.281,108721.302,109852.111,114817.737,122809.510,138844.393,152279.608,166686.215,ALAGOINHA,Mil Reais
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219,MU,2516904,Uiraúna (PB),83298.629,91020.702,105395.082,114231.948,134146.063,141785.040,156624.916,162923.714,164775.616,173817.558,178335.737,193070.808,219080.488,245509.735,UIRAUNA,Mil Reais
220,MU,2517001,Umbuzeiro (PB),41516.718,47125.483,53501.923,57995.250,63156.027,66492.576,69532.617,72567.138,74204.147,76567.049,83714.748,91269.936,95803.915,118557.079,UMBUZEIRO,Mil Reais
221,MU,2517100,Várzea (PB),16485.698,18651.469,20408.239,22730.828,24743.383,27336.097,27175.268,29715.289,30164.984,31479.072,34060.903,37673.128,41254.581,46408.077,VARZEA,Mil Reais
222,MU,2517209,Vieirópolis (PB),20802.329,23743.339,25834.494,29529.374,32399.003,35704.624,38292.979,41395.854,41975.766,41818.690,45288.539,52850.640,49371.042,59242.478,VIEIROPOLIS,Mil Reais


## 7. Participações do PIB municipal e CR2

In [7]:
# Usa o total da Paraíba publicado na tabela municipal (mesma fonte e unidade).
participacoes = []
if len(total_pb_mun) == 1:
    total = total_pb_mun.iloc[0]
    for _, r in municipios.iterrows():
        for ano in ANOS:
            valor = r.get(str(ano), np.nan)
            denominador = total.get(str(ano), np.nan)
            pct = valor / denominador * 100 if pd.notna(valor) and pd.notna(denominador) and denominador != 0 else np.nan
            participacoes.append({
                "codigo_ibge": r["codigo_ibge"], "municipio": r["municipio"],
                "municipio_padronizado": r["municipio_padronizado"],
                "ano": ano, "pib_mil_reais": valor,
                "pib_total_paraiba_mil_reais": denominador,
                "participacao_pib_pb_pct": pct
            })
participacoes = pd.DataFrame(participacoes)

if not participacoes.empty:
    ranking = participacoes.dropna(subset=["participacao_pib_pb_pct"]).sort_values(
        ["ano","participacao_pib_pb_pct"], ascending=[True,False])
    top2 = ranking.groupby("ano").head(2)
    cr2 = top2.groupby("ano", as_index=False)["participacao_pib_pb_pct"].sum().rename(
        columns={"participacao_pib_pb_pct":"CR2_pct"})
    print("Participações:", participacoes.shape, "| anos com CR2:", len(cr2))
    display(cr2)
else:
    cr2 = pd.DataFrame(columns=["ano","CR2_pct"])
    print("Não foi possível calcular participações: total estadual municipal ausente ou duplicado.")

Participações: (3122, 7) | anos com CR2: 14


,ano,CR2_pct
0,2010,47.242670
1,2011,46.953603
2,2012,47.843753
3,2013,46.524301
4,2014,47.197395
5,2015,46.718412
6,2016,45.874496
7,2017,45.500439
8,2018,44.991887
9,2019,44.646858


## 8. Tratamento do ICMS a partir dos PDFs

Os PDFs são relatórios mensais. Nos arquivos de 2015–2017 e 2021–2024, o valor total aparece como `TOTAL DA ARRECADAÇÃO DO ICMS`. Em 2025–2026, aparece como `Arrecadação ICMS`. Para documentos escaneados, o notebook tenta OCR (Tesseract em português). Se o relatório não contiver o quadro/valor, o mês fica marcado como não extraído — não se preenche por estimativa.

In [8]:
# Dependências para leitura de PDFs e OCR de documentos escaneados (Google Colab)
!pip -q install pdfplumber pytesseract xlrd openpyxl
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-por
print("Dependências de PDF/OCR instaladas.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package tesseract-ocr-por.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-por_1%3a4.1.0-2_all.deb ...
Unpacking tesseract-ocr-por (1:4.1.0-2) ...
Setting up tesseract-ocr-por (1:4.1.0-2) ...
Dependências de PDF/OCR instaladas.


In [9]:
import pdfplumber
from PIL import Image
import pytesseract
OCR_SCANNED = True  # Altere para False se quiser pular OCR de páginas digitalizadas

def extrair_texto_pagina(page):
    txt = page.extract_text() or ""
    if len(txt.strip()) < 80:
        try:
            img = page.to_image(resolution=220).original
            txt_ocr = pytesseract.image_to_string(img, lang="por", timeout=25) if OCR_SCANNED else ""
            if len(txt_ocr.strip()) > len(txt.strip()): txt = txt_ocr
        except Exception as e:
            pass
    return txt

def parse_valor_br(s):
    return ptbr_num(s)

icms_registros = []
icms_paginas = []
for path in pdf_paths:
    m = re.search(r"ICMS(20\d{2})", path.name, flags=re.I)
    if not m: continue
    ano_arquivo = int(m.group(1))
    with pdfplumber.open(path) as pdf:
        for i, page in enumerate(pdf.pages):
            txt = extrair_texto_pagina(page)
            data = re.search(r"Per[ií]odo:\s*(\d{2})/(\d{2})/(\d{4})", txt, flags=re.I)
            if not data:
                icms_paginas.append({"arquivo":path.name,"pagina":i+1,"status":"sem data/período extraível"})
                continue
            mes, ano = int(data.group(2)), int(data.group(3))
            valor_match = re.search(r"TOTAL DA ARRECADA[CÇ][AÃ]O DO ICMS\s*([\d.]+,\d{2})", txt, flags=re.I)
            if not valor_match:
                valor_match = re.search(r"Arrecada[cç][aã]o ICMS\s+([\d.]+,\d{2})", txt, flags=re.I)
            valor = parse_valor_br(valor_match.group(1)) if valor_match else np.nan
            status = "extraído" if pd.notna(valor) else "período identificado, total não localizado"
            icms_registros.append({
                "ano":ano,"mes":mes,"data_inicio":f"{ano:04d}-{mes:02d}-01",
                "icms_total_reais":valor,"arquivo_fonte":path.name,
                "pagina_pdf":i+1,"status_extracao":status
            })
            icms_paginas.append({"arquivo":path.name,"pagina":i+1,"ano":ano,"mes":mes,"status":status})

icms_mensal = pd.DataFrame(icms_registros)
if not icms_mensal.empty:
    # 2025/26 têm várias páginas por mês; mantém uma ocorrência por ano/mês com valor válido.
    icms_mensal = (icms_mensal.sort_values("status_extracao")
                   .drop_duplicates(["ano","mes"], keep="first")
                   .sort_values(["ano","mes"]).reset_index(drop=True))
icms_paginas = pd.DataFrame(icms_paginas)
print("Meses/períodos identificados:", len(icms_mensal))
print("Meses com valor ICMS extraído:", int(icms_mensal["icms_total_reais"].notna().sum()) if not icms_mensal.empty else 0)
display(icms_mensal)
display(icms_paginas[icms_paginas["status"]!="extraído"] if not icms_paginas.empty else icms_paginas)

Meses/períodos identificados: 141
Meses com valor ICMS extraído: 109


,ano,mes,data_inicio,icms_total_reais,arquivo_fonte,pagina_pdf,status_extracao
0,2015,1,2015-01-01,4.364769e+08,ICMS2015.pdf,1,extraído
1,2015,2,2015-02-01,3.675061e+08,ICMS2015.pdf,2,extraído
2,2015,3,2015-03-01,3.457801e+08,ICMS2015.pdf,3,extraído
3,2015,4,2015-04-01,3.851358e+08,ICMS2015.pdf,4,extraído
4,2015,5,2015-05-01,3.844080e+08,ICMS2015.pdf,5,extraído
...,...,...,...,...,...,...,...
136,2026,5,2026-05-01,9.097283e+08,ICMS2026.pdf,17,extraído
137,2026,6,2026-06-01,9.386976e+08,ICMS2026.pdf,21,extraído
138,2026,7,2026-07-01,9.739098e+08,ICMS2026.pdf,25,extraído
139,2026,8,2026-08-01,9.547024e+08,ICMS2026.pdf,29,extraído


,arquivo,pagina,ano,mes,status
36,ICMS2018.pdf,1,2018,1,"período identificado, total não localizado"
37,ICMS2018.pdf,2,2018,2,"período identificado, total não localizado"
38,ICMS2018.pdf,3,2018,3,"período identificado, total não localizado"
40,ICMS2018.pdf,5,2018,5,"período identificado, total não localizado"
41,ICMS2018.pdf,6,2018,6,"período identificado, total não localizado"
...,...,...,...,...,...
198,ICMS2026.pdf,31,2026,8,"período identificado, total não localizado"
199,ICMS2026.pdf,32,2026,8,"período identificado, total não localizado"
201,ICMS2026.pdf,34,2026,9,"período identificado, total não localizado"
202,ICMS2026.pdf,35,2026,9,"período identificado, total não localizado"


## 9. Outras bases CSV e planilhas

In [10]:
# Leitura conservadora: preservar tabelas não PIB em sua estrutura original, com cabeçalho identificado.
outras_bases = {}
for path in csv_paths:
    if "pib" in path.name.lower(): continue
    try:
        # IBGE de desocupação e PMC: cabeçalho de períodos na linha 3.
        if any(t in path.name.lower() for t in ["desocupacao","desocupação","pmc"]):
            df = pd.read_csv(path, sep=";", header=3, dtype=str, encoding="utf-8-sig",
                             engine="python", index_col=False, on_bad_lines="warn")
        elif "exportacao" in path.name.lower() or "exportação" in path.name.lower():
            df = pd.read_csv(path, sep=";", header=0, dtype=str, encoding="utf-8-sig",
                             engine="python", index_col=False, on_bad_lines="warn")
        else:
            df = pd.read_csv(path, sep=None, engine="python", dtype=str, encoding="utf-8-sig")
        outras_bases[path.stem] = df
        safe = re.sub(r"[^A-Za-z0-9_-]+","_",path.stem)
        df.to_csv(PROC/f"{safe}_preservada.csv", index=False, encoding="utf-8-sig")
        print(path.name, df.shape)
    except Exception as e:
        print("Falha de leitura",path.name,repr(e))

excel_status = []
for path in excel_paths:
    try:
        xls = pd.ExcelFile(path)
        for sheet in xls.sheet_names:
            df = pd.read_excel(path, sheet_name=sheet, dtype=object)
            safe = re.sub(r"[^A-Za-z0-9_-]+","_",f"{path.stem}_{sheet}")
            df.to_csv(PROC/f"{safe}_preservada.csv",index=False,encoding="utf-8-sig")
            excel_status.append({"arquivo":path.name,"aba":sheet,"linhas":len(df),"colunas":len(df.columns),"status":"lido"})
    except Exception as e:
        excel_status.append({"arquivo":path.name,"aba":None,"status":f"não lido: {e}"})
display(pd.DataFrame(excel_status))

V_EXPORTACAO_E IMPORTACAO_GERAL_2010-01_2026-12_DT20261105.csv (34, 5)
tabela4099.desocupaçãoPB.csv (12, 60)
tabela8880.PMCparaíba.csv (11, 142)
Falha de leitura tabela8881.PMSparaíba.csv ParserError('\' \' expected after \'"\'')


,arquivo,aba,linhas,colunas,status
0,4-tabelas.xls,CAGED,20,2,lido
1,4-tabelas.xls,Sumário,21,3,lido
2,4-tabelas.xls,tabela1,49,13,lido
3,4-tabelas.xls,tabela2,41,13,lido
4,4-tabelas.xls,tabela3,35,5,lido
5,4-tabelas.xls,tabela4,37,29,lido
6,4-tabelas.xls,tabela4.1,42,256,lido
7,4-tabelas.xls,tabela4.2,24,29,lido
8,4-tabelas.xls,tabela5,71,29,lido
9,4-tabelas.xls,tabela6,49,9,lido


## 10. Testes de integridade e consistência

In [11]:
testes = []
def teste(nome, ok, detalhe):
    testes.append({"teste":nome,"resultado":"OK" if bool(ok) else "VERIFICAR","detalhe":str(detalhe)})

teste("223 municípios na tabela IBGE", len(municipios)==223, f"Encontrados: {len(municipios)}")
teste("Código IBGE municipal com 7 dígitos", municipios["codigo_ibge"].str.fullmatch(r"\d{7}").all(),
      f"Inválidos: {(~municipios['codigo_ibge'].str.fullmatch(r'\d{7}')).sum()}")
teste("Código IBGE único", municipios["codigo_ibge"].is_unique, f"Únicos: {municipios['codigo_ibge'].nunique()}")
teste("Total estadual municipal único", len(total_pb_mun)==1, f"Linhas UF: {len(total_pb_mun)}")
teste("Participações calculadas", not participacoes.empty, f"Registros: {len(participacoes)}")
if not participacoes.empty:
    soma = participacoes.groupby("ano")["participacao_pib_pb_pct"].sum()
    teste("Soma participações municipais próxima de 100%", ((soma-100).abs()<1).all(), soma.round(3).to_dict())
teste("ICMS com pelo menos um mês extraído", not icms_mensal.empty and icms_mensal["icms_total_reais"].notna().any(),
      f"Meses com valor: {int(icms_mensal['icms_total_reais'].notna().sum()) if not icms_mensal.empty else 0}")
relatorio_testes = pd.DataFrame(testes)
display(relatorio_testes)

,teste,resultado,detalhe
0,223 municípios na tabela IBGE,OK,Encontrados: 223
1,Código IBGE municipal com 7 dígitos,OK,Inválidos: 0
2,Código IBGE único,OK,Únicos: 223
3,Total estadual municipal único,OK,Linhas UF: 1
4,Participações calculadas,OK,Registros: 3122
5,Soma participações municipais próxima de 100%,OK,"{2010: 100.0, 2011: 100.0, 2012: 100.0, 2013: ..."
6,ICMS com pelo menos um mês extraído,OK,Meses com valor: 109


## 11. Relatório de qualidade e dicionário

In [12]:
qualidade = pd.DataFrame([
 {"base/variável":"PIB Brasil, estados e Paraíba","problema":"Metadados nas primeiras linhas",
  "decisão":"Leitura do cabeçalho na linha 4 e conversão dos anos para numérico",
  "validação":"Dimensões e valores anuais conferidos após leitura."},
 {"base/variável":"PIB municipal PB","problema":"Colunas de unidade intercaladas com os anos; linhas de notas e legenda",
  "decisão":"Leitura com separador ; e index_col=False; seleção de UF e MU; unidade Mil Reais",
  "validação":f"{len(municipios)} municípios e {municipios['codigo_ibge'].nunique()} códigos únicos."},
 {"base/variável":"Participação PIB / CR2","problema":"Necessidade de denominador estadual compatível",
  "decisão":"PIB municipal dividido pelo total estadual da mesma tabela e ano",
  "validação":f"{len(cr2)} anos com CR2 calculado."},
 {"base/variável":"ICMS PDFs","problema":"Relatórios variam entre anos; alguns PDFs são escaneados e certos meses não contêm quadro extraível",
  "decisão":"Extração de texto/OCR; ausência de valor fica como ausente, sem imputação",
  "validação":f"{int(icms_mensal['icms_total_reais'].notna().sum()) if not icms_mensal.empty else 0} meses com valor extraído."},
])
display(qualidade)

dicionario = pd.DataFrame([
 {"base":"pib_municipios","variavel":"codigo_ibge","descricao":"Código IBGE municipal","unidade":"identificador"},
 {"base":"pib_municipios","variavel":"municipio","descricao":"Nome original da fonte","unidade":"texto"},
 {"base":"pib_municipios","variavel":"2010–2023","descricao":"PIB municipal a preços correntes","unidade":"Mil Reais"},
 {"base":"participacoes_pib","variavel":"participacao_pib_pb_pct","descricao":"PIB municipal / PIB total PB × 100","unidade":"%"},
 {"base":"CR2","variavel":"CR2_pct","descricao":"Soma das participações dos dois maiores municípios no ano","unidade":"%"},
 {"base":"icms_mensal","variavel":"icms_total_reais","descricao":"Total mensal da arrecadação do ICMS, conforme relatório","unidade":"R$"},
])
display(dicionario)

,base/variável,problema,decisão,validação
0,"PIB Brasil, estados e Paraíba",Metadados nas primeiras linhas,Leitura do cabeçalho na linha 4 e conversão do...,Dimensões e valores anuais conferidos após lei...
1,PIB municipal PB,Colunas de unidade intercaladas com os anos; l...,Leitura com separador ; e index_col=False; sel...,223 municípios e 223 códigos únicos.
2,Participação PIB / CR2,Necessidade de denominador estadual compatível,PIB municipal dividido pelo total estadual da ...,14 anos com CR2 calculado.
3,ICMS PDFs,Relatórios variam entre anos; alguns PDFs são ...,Extração de texto/OCR; ausência de valor fica ...,109 meses com valor extraído.


,base,variavel,descricao,unidade
0,pib_municipios,codigo_ibge,Código IBGE municipal,identificador
1,pib_municipios,municipio,Nome original da fonte,texto
2,pib_municipios,2010–2023,PIB municipal a preços correntes,Mil Reais
3,participacoes_pib,participacao_pib_pb_pct,PIB municipal / PIB total PB × 100,%
4,CR2,CR2_pct,Soma das participações dos dois maiores municí...,%
5,icms_mensal,icms_total_reais,"Total mensal da arrecadação do ICMS, conforme ...",R$


## 12. Exportação das bases finais e documentação

In [13]:
# Bases PIB
pib_brasil.to_csv(PROC/"pib_brasil_tratado.csv",index=False,encoding="utf-8-sig")
pib_estados.to_csv(PROC/"pib_estados_tratado.csv",index=False,encoding="utf-8-sig")
pib_paraiba.to_csv(PROC/"pib_paraiba_tratado.csv",index=False,encoding="utf-8-sig")
pib_mun.to_csv(PROC/"pib_municipios_pb_tratado.csv",index=False,encoding="utf-8-sig")
municipios.to_csv(PROC/"pib_municipios_pb_somente_municipios.csv",index=False,encoding="utf-8-sig")
participacoes.to_csv(PROC/"participacoes_pib_municipal.csv",index=False,encoding="utf-8-sig")
cr2.to_csv(PROC/"indicador_CR2.csv",index=False,encoding="utf-8-sig")

# ICMS e diagnósticos
icms_mensal.to_csv(PROC/"icms_mensal_extraido.csv",index=False,encoding="utf-8-sig")
icms_paginas.to_csv(DOCS/"diagnostico_paginas_icms.csv",index=False,encoding="utf-8-sig")
relatorio_testes.to_csv(DOCS/"testes_integridade.csv",index=False,encoding="utf-8-sig")
qualidade.to_csv(DOCS/"relatorio_qualidade.csv",index=False,encoding="utf-8-sig")
dicionario.to_csv(DOCS/"dicionario_dados.csv",index=False,encoding="utf-8-sig")

readme = f'''# E05 — Dados processados
Execução: {EXECUCAO}

## Estrutura
- data/raw: ZIP original
- data/raw/extraidos: arquivos originais extraídos
- data/processed: bases tratadas e indicadores
- docs: inventário, dicionário, testes e relatório de qualidade

## Resultados
- Municípios na tabela PIB: {len(municipios)}
- Códigos IBGE únicos: {municipios["codigo_ibge"].nunique()}
- Registros de participação PIB: {len(participacoes)}
- Anos com CR2: {len(cr2)}
- Meses com ICMS total extraído: {int(icms_mensal["icms_total_reais"].notna().sum()) if not icms_mensal.empty else 0}

## Limitações
O OCR e a extração de PDF podem não recuperar todos os meses. Consulte docs/diagnostico_paginas_icms.csv. Não foram imputados valores ausentes.
'''
(DOCS/"README.md").write_text(readme,encoding="utf-8")

print("Exportação concluída.")
print("Processados:", [p.name for p in PROC.glob("*")])
print("Documentação:", [p.name for p in DOCS.glob("*")])

Exportação concluída.
Processados: ['4-tabelas_tabela4_2_preservada.csv', 'icms_mensal_extraido.csv', 'indicador_CR2.csv', '4-tabelas_tabela4_preservada.csv', '4-tabelas_Tabela9_preservada.csv', 'participacoes_pib_municipal.csv', 'tabela4099_desocupa_oPB_preservada.csv', 'tabela8880_PMCpara_ba_preservada.csv', '4-tabelas_tabela2_preservada.csv', 'PB_grafico_preservada.csv', '4-tabelas_CAGED_preservada.csv', 'pib_brasil_tratado.csv', '4-tabelas_tabela6_preservada.csv', 'pib_municipios_pb_somente_municipios.csv', 'PB_CAGED_preservada.csv', '4-tabelas_tabela7_1_preservada.csv', 'pib_municipios_pb_tratado.csv', 'V_EXPORTACAO_E_IMPORTACAO_GERAL_2010-01_2026-12_DT20261105_preservada.csv', '4-tabelas_tabela11_preservada.csv', '4-tabelas_tabela9_1_preservada.csv', '4-tabelas_tabela10_1_preservada.csv', '4-tabelas_tabela1_preservada.csv', '4-tabelas_tabela5_preservada.csv', '4-tabelas_tabela8_preservada.csv', '4-tabelas_tabela7_preservada.csv', 'PB_municipios_preservada.csv', '4-tabelas_tabela1

## 13. Resumo final
Confira as tabelas de testes e o relatório de qualidade antes da submissão. A entrega do guia exige execução de ponta a ponta, preservação dos originais, validação, dicionário e README. Este notebook registra os casos em que a fonte não permite extração automática completa; eles precisam ser revisados pela equipe.

In [14]:
print("E05 — RESUMO FINAL")
print("ZIP:", zip_path)
print("Arquivos originais inventariados:", len(inventario))
print("Municípios PIB:", len(municipios))
print("Participações PIB:", len(participacoes))
print("CR2 anos:", len(cr2))
print("Meses ICMS com valor extraído:", int(icms_mensal["icms_total_reais"].notna().sum()) if not icms_mensal.empty else 0)
print("\nTestes:")
display(relatorio_testes)
print("Pasta de entrega:", ROOT.resolve())

E05 — RESUMO FINAL
ZIP: E05/data/raw/todos  os dados-20261105T232855Z-1-001.zip
Arquivos originais inventariados: 22
Municípios PIB: 223
Participações PIB: 3122
CR2 anos: 14
Meses ICMS com valor extraído: 109

Testes:


,teste,resultado,detalhe
0,223 municípios na tabela IBGE,OK,Encontrados: 223
1,Código IBGE municipal com 7 dígitos,OK,Inválidos: 0
2,Código IBGE único,OK,Únicos: 223
3,Total estadual municipal único,OK,Linhas UF: 1
4,Participações calculadas,OK,Registros: 3122
5,Soma participações municipais próxima de 100%,OK,"{2010: 100.0, 2011: 100.0, 2012: 100.0, 2013: ..."
6,ICMS com pelo menos um mês extraído,OK,Meses com valor: 109


Pasta de entrega: /content/E05
